<a href="https://colab.research.google.com/github/182Marco/esercizi-Python-boolean/blob/main/Pa_Progetto_Guida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Progetto Riepilogo M1 — AI Help Desk 🎫

> **Modulo M1: Python & Intro AI** — Corso **AI Engineering** (Boolean)

Otto lezioni di Python, un solo progetto: il **triage di un help desk**.
Un programma che legge ticket di supporto da un file JSON (alcuni
volutamente "sporchi"), li **valida**, li **classifica** per categoria,
calcola statistiche, scrive un **report** ed evade i ticket urgenti
**in parallelo**.

È lo scheletro di una vera applicazione AI: alla fine di questo notebook
vediamo in anteprima come la classificazione potrebbe farla un **LLM**,
con una vera chiamata all'API di OpenAI.

## La mappa: ogni lezione ha un posto nel progetto

| Lezione | Argomento | Dove lo usi nel progetto |
|---|---|---|
| 1 | f-strings, comprehensions, type hints | report, statistiche, firme delle funzioni |
| 2 | liste, dizionari, set | raccolta ticket, conteggi per categoria, tag unici |
| 3 | `*args`/`**kwargs`, lambda, pathlib | helper di report, ordinamenti, percorsi dei file |
| 4 | classi | il classificatore è una classe con stato e metodi |
| 5 | ereditarietà, dataclass, ABC | `ClassificatoreBase` (ABC) → `ClassificatoreParoleChiave`; `RisultatoTriage` |
| 6 | try/except, context manager, JSON/YAML | caricamento dati sporchi, `config.yaml`, scrittura report |
| 7 | Pydantic | schema `Ticket`, scarto dei record non validi |
| 8 | async/await, httpx | evasione in parallelo degli urgenti, demo OpenAI |

## 1. Diamo un'occhiata ai dati

In `dati/tickets.json` ci sono **12 ticket**. Attenzione: come nella vita
reale, **non tutti sono validi** (campi mancanti, priorità impossibili,
tipi sbagliati). Sarà compito vostro scartarli.

In [ ]:
import json
from pathlib import Path

tickets_grezzi = json.loads(Path("dati/tickets.json").read_text(encoding="utf-8"))

print(f"{len(tickets_grezzi)} ticket nel file. Il primo:")
print(tickets_grezzi[0])

In [ ]:
# Le regole del triage stanno in un file di configurazione YAML (Lezione 6)
import yaml

config = yaml.safe_load(Path("dati/config.yaml").read_text(encoding="utf-8"))

for categoria, parole in config["categorie"].items():
    print(f"{categoria:14} -> parole chiave: {', '.join(parole)}")
print(f"{'(default)':14} -> {config['categoria_default']}")
print(f"Urgente se priorita >= {config['soglia_urgenza']}")

## 2. Il pipeline da costruire

```
tickets.json ──> [1. carica]          try/except, with open, json      (Lez. 6)
                 [2. valida]          Pydantic: 9 validi, 3 scartati   (Lez. 7)
                 [3. classifica]      ABC + classe con parole chiave   (Lez. 4-5)
                 [4. statistiche]     dict/set comprehension, lambda   (Lez. 1-3)
                 [5. report]          f-strings, with open, JSON       (Lez. 1, 6)
                 [6. evadi urgenti]   asyncio.gather in parallelo      (Lez. 8)
```

La consegna completa è nel `README.md`. Si lavora su
**`Pb Progetto Starter.ipynb`**: ogni passo ha dei `TODO` e dei
**self-test** (`assert`) — se la cella passa, il passo è corretto.
La soluzione (`Pc`) si guarda **solo dopo** averci provato. 😉

---

## 3. Anteprima della prossima lezione: e se classificasse un LLM? 🤖

Il nostro `ClassificatoreParoleChiave` è fragile: *"Non mi è mai arrivato
il collo"* non contiene nessuna parola chiave, ma un umano capirebbe
subito che è una spedizione.

La buona notizia: una chiamata a un LLM è **solo una richiesta HTTP con un
JSON** — esattamente le cose viste nelle lezioni 6 e 8. Nessuna magia:

- un **endpoint**: `https://api.openai.com/v1/chat/completions`
- un header con la **API key** (la "password" del servizio)
- un **JSON** con il modello da usare e i messaggi
- la risposta è di nuovo un **JSON** da cui estrarre il testo

La cella qui sotto fa una chiamata vera **solo se** trova la variabile
d'ambiente `OPENAI_API_KEY`, altrimenti si salta con un messaggio.

In [ ]:
import os
import httpx

api_key = os.getenv("OPENAI_API_KEY")
messaggio = "Il pacco risulta consegnato ma non ho ricevuto nulla."

if not api_key:
    print("Nessuna OPENAI_API_KEY nell'ambiente: demo saltata.")
    print("Per provarla:  $env:OPENAI_API_KEY = 'sk-...'  e riavvia Jupyter da quel terminale.")
else:
    risposta = httpx.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "gpt-4o-mini",
            "messages": [
                {"role": "system", "content": "Classifica il ticket con una sola parola: "
                                              "tecnico, fatturazione, spedizione o generale."},
                {"role": "user", "content": messaggio},
            ],
        },
        timeout=30,
    )
    dati = risposta.json()
    print("Ticket :", messaggio)
    print("LLM    :", dati["choices"][0]["message"]["content"])

Output atteso (con la chiave impostata):

```text
Ticket : Il pacco risulta consegnato ma non ho ricevuto nulla.
LLM    : spedizione
```

> 💰 **Nota costi**: una chiamata così costa **frazioni di centesimo**
> (`gpt-4o-mini` è il modello economico di OpenAI). La chiave si crea su
> <https://platform.openai.com> — la configureremo insieme in questa
> lezioni, oggi è solo un assaggio.

### 📘 Riferimento (non eseguire ora): la stessa chiamata con l'SDK ufficiale

Dalla lezione 10 in poi useremo la libreria ufficiale `openai`, che nasconde
i dettagli HTTP che abbiamo appena visto:

```python
# pip install openai
from openai import OpenAI

client = OpenAI()  # legge OPENAI_API_KEY dall'ambiente

risposta = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Classifica il ticket con una sola parola: "
                                      "tecnico, fatturazione, spedizione o generale."},
        {"role": "user", "content": "Il pacco risulta consegnato ma non ho ricevuto nulla."},
    ],
)
print(risposta.choices[0].message.content)
```

E grazie alla nostra ABC, un classificatore basato su LLM è solo
**un'altra sottoclasse** — il resto del programma non cambia di una riga:

```python
class ClassificatoreLLM(ClassificatoreBase):
    """Stesso contratto di ClassificatoreParoleChiave, ma decide un LLM."""

    def classifica(self, ticket: Ticket) -> str:
        risposta = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Classifica il ticket con una sola parola: "
                                              "tecnico, fatturazione, spedizione o generale."},
                {"role": "user", "content": ticket.messaggio},
            ],
        )
        return risposta.choices[0].message.content.strip().lower()
```

Questo è il motivo per cui abbiamo progettato il triage attorno a un'ABC:
**cambia il "cervello", non l'architettura**.

---

## Prossimi passi

1. Completa **`Pb Progetto Starter.ipynb`** (la consegna è nel `README.md`).
2. Nella prossima lezione iniziamo il blocco AI: **cos'è un LLM**,
   l'architettura Transformer ad alto livello, la tokenization e la
   context window. E da lì in poi, l'API che avete appena visto diventerà
   il vostro strumento di lavoro quotidiano.